# Olist E-Commerce Data Pipeline
**Author:** Amirabas Ziaee
**GitHub:** github.com/aaziaee
**LinkedIn:** linkedin.com/in/amirabasziaee

Loads the Olist Brazilian E-Commerce dataset into a normalized
PostgreSQL schema using SQLAlchemy, with data quality validation
at each stage.

## Setup

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from getpass import getpass

load_dotenv()  # loads variables from a local .env file, if present (see .env.example)

## Database Connection
Connection settings are read from environment variables (see `.env.example`),
falling back to sensible local defaults: `OLIST_DB_HOST`, `OLIST_DB_PORT`,
`OLIST_DB_NAME`, `OLIST_DB_USER`, `OLIST_DB_PASSWORD`. The password is never
hardcoded — it's read from `OLIST_DB_PASSWORD` if set, or prompted for
interactively otherwise.

In [2]:
db_host = os.environ.get("OLIST_DB_HOST", "localhost")
db_port = os.environ.get("OLIST_DB_PORT", "5432")
db_name = os.environ.get("OLIST_DB_NAME", "olist_ecommerce")
db_user = os.environ.get("OLIST_DB_USER", "postgres")
db_password = os.environ.get("OLIST_DB_PASSWORD") or getpass("Database password: ")

engine = create_engine(
    f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

## Load Configuration
`DATA_DIR` points to the folder containing the raw Olist CSVs (not included
in this repo — see the main README for the dataset source). Override with
the `OLIST_DATA_DIR` environment variable if your CSVs live elsewhere;
defaults to a `data/` folder at the repo root.

In [3]:
DATA_DIR = Path(os.environ.get("OLIST_DATA_DIR", "../data"))

load_order = [
    "product_category_name_translation",
    "sellers",
    "customers",
    "geolocation",
    "products",
    "orders",
    "order_items",
    "order_payments",
    "order_reviews"
] 

csv_files = {
    "product_category_name_translation": "product_category_name_translation",
    "sellers": "olist_sellers_dataset",
    "customers": "olist_customers_dataset",
    "geolocation": "olist_geolocation_dataset",
    "products": "olist_products_dataset",
    "orders": "olist_orders_dataset",
    "order_items": "olist_order_items_dataset",
    "order_payments": "olist_order_payments_dataset",
    "order_reviews": "olist_order_reviews_dataset"
} 

date_columns = {
    "product_category_name_translation": [],
    "sellers": [],
    "customers": [],
    "geolocation": [],
    "products": [],
    "orders": ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"],   
    "order_items": ["shipping_limit_date", ],
    "order_payments": [],
    "order_reviews": ["review_creation_date", "review_answer_timestamp"],
}

## Data Quality Fix — Orphan Product Categories
2 categories present in `olist_products_dataset.csv` have no matching
row in `product_category_name_translation.csv`: `pc_gamer` and
`portateis_cozinha_e_preparadores_de_alimentos`. Without this fix, the
products load fails on the foreign key constraint. English name left
NULL rather than inventing a translation the source data never provided.

In [4]:
with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO product_category_name_translation
            (product_category_name, product_category_name_english)
        VALUES
            ('pc_gamer', NULL),
            ('portateis_cozinha_e_preparadores_de_alimentos', NULL)
    """))
    conn.commit()

## Full Load

In [5]:
for table in load_order:
    try:
        filename = csv_files[table]
        df = pd.read_csv(
            DATA_DIR / f"{filename}.csv",
            parse_dates=date_columns[table],
            date_format="%m/%d/%Y %H:%M"
        )
        df.to_sql(table, con=engine, if_exists="append", index=False)
        print(f"{table}: {len(df)} rows loaded")
    except Exception as e:
        print(f"{table}: FAILED — {e}")
        break

product_category_name_translation: 71 rows loaded
sellers: 3095 rows loaded
customers: 99441 rows loaded
geolocation: 1000163 rows loaded
products: 32951 rows loaded
orders: 99441 rows loaded
order_items: 112650 rows loaded
order_payments: 103886 rows loaded
order_reviews: 99224 rows loaded


## Post-Load Verification

In [6]:
for table in load_order:
    result = pd.read_sql(text(f"SELECT COUNT(*) FROM {table}"), con=engine)
    print(f"{table}: {result.iloc[0,0]} rows")

product_category_name_translation: 73 rows
sellers: 3095 rows
customers: 99441 rows
geolocation: 1000163 rows
products: 32951 rows
orders: 99441 rows
order_items: 112650 rows
order_payments: 103886 rows
order_reviews: 99224 rows


## Encoding Check
Confirms accented Portuguese text (e.g. "não", "são") in free-text
review comments survived the load correctly.

In [7]:
sample_reviews = pd.read_sql(
    text("SELECT review_comment_message FROM order_reviews WHERE review_comment_message LIKE '%ã%' LIMIT 3"),
    con=engine
)
sample_reviews

,review_comment_message
0,Não gostei ! Comprei gato por lebre
1,Sempre compro pela Internet e a entrega ocorre...
2,O produto não chegou no prazo estipulado e cau...


## Cleanup

In [8]:
engine.dispose()